# AI Ops Automation Notebook

End-to-end AI Ops pipeline: Jira ingestion → classification → log fetching → RCA generation → notifications.

This notebook combines all modules from the `aiops` package:
1. **Configuration** (`config.py`)
2. **Celery App** (`celery_app.py`)
3. **Model Loader** (`models/loader.py`)
4. **Classify Task** (`tasks/classify.py`)
5. **Log Fetch Task** (`tasks/log_fetch.py`)
6. **Search Task** (`tasks/search.py`)
7. **RCA Task** (`tasks/rca.py`)
8. **Notify Task** (`tasks/notify.py`)
9. **Confluence KB Task** (`tasks/confluence_kb.py`)
10. **FastAPI Application** (`main.py`)

## 1. Install Dependencies

In [ ]:
# Uncomment and run to install required packages
# !pip install -r aiops/requirements.txt

## 2. Configuration (`config.py`)

Central configuration loaded from environment variables.

In [ ]:
"""Central configuration loaded from environment variables."""

from __future__ import annotations

import os
from dotenv import load_dotenv

load_dotenv()


def _require(key: str) -> str:
    val = os.getenv(key)
    if not val:
        raise RuntimeError(f"Required environment variable '{key}' is not set.")
    return val


# ---------------------------------------------------------------------------
# Jira
# ---------------------------------------------------------------------------
JIRA_BASE_URL: str = os.getenv("JIRA_BASE_URL", "")          # e.g. https://myorg.atlassian.net
JIRA_USER: str = os.getenv("JIRA_USER", "")                   # service-account email
JIRA_API_TOKEN: str = os.getenv("JIRA_API_TOKEN", "")
JIRA_PROJECT_KEY: str = os.getenv("JIRA_PROJECT_KEY", "OPS")

# ---------------------------------------------------------------------------
# Confluence
# ---------------------------------------------------------------------------
CONFLUENCE_BASE_URL: str = os.getenv("CONFLUENCE_BASE_URL", JIRA_BASE_URL)
CONFLUENCE_SPACE_KEY: str = os.getenv("CONFLUENCE_SPACE_KEY", "KB")
CONFLUENCE_PARENT_PAGE_ID: str = os.getenv("CONFLUENCE_PARENT_PAGE_ID", "")

# ---------------------------------------------------------------------------
# Azure Storage (mobile logs)
# ---------------------------------------------------------------------------
AZURE_STORAGE_CONNECTION_STRING: str = os.getenv("AZURE_STORAGE_CONNECTION_STRING", "")
AZURE_FILE_SHARE_NAME: str = os.getenv("AZURE_FILE_SHARE_NAME", "mobile-logs")
AZURE_LOG_DIRECTORY: str = os.getenv("AZURE_LOG_DIRECTORY", "")

# ---------------------------------------------------------------------------
# Datadog (server logs + monitoring)
# ---------------------------------------------------------------------------
DATADOG_API_KEY: str = os.getenv("DATADOG_API_KEY", "")
DATADOG_APP_KEY: str = os.getenv("DATADOG_APP_KEY", "")
DATADOG_SITE: str = os.getenv("DATADOG_SITE", "datadoghq.com")
DATADOG_LOG_LOOKBACK_HOURS: int = int(os.getenv("DATADOG_LOG_LOOKBACK_HOURS", "6"))

# ---------------------------------------------------------------------------
# OpenAI / Azure OpenAI
# ---------------------------------------------------------------------------
OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL: str = os.getenv("OPENAI_MODEL", "gpt-4o")
AZURE_OPENAI_ENDPOINT: str = os.getenv("AZURE_OPENAI_ENDPOINT", "")
AZURE_OPENAI_API_KEY: str = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_DEPLOYMENT: str = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
AZURE_OPENAI_API_VERSION: str = os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-01")

# ---------------------------------------------------------------------------
# Email / SMTP
# ---------------------------------------------------------------------------
SMTP_HOST: str = os.getenv("SMTP_HOST", "smtp.gmail.com")
SMTP_PORT: int = int(os.getenv("SMTP_PORT", "587"))
SMTP_USER: str = os.getenv("SMTP_USER", "")
SMTP_PASSWORD: str = os.getenv("SMTP_PASSWORD", "")
ALERT_EMAIL_RECIPIENTS: list[str] = [
    e.strip()
    for e in os.getenv("ALERT_EMAIL_RECIPIENTS", "").split(",")
    if e.strip()
]

# ---------------------------------------------------------------------------
# Celery / Redis
# ---------------------------------------------------------------------------
REDIS_URL: str = os.getenv("REDIS_URL", "redis://localhost:6379/0")

# ---------------------------------------------------------------------------
# Model artifacts
# ---------------------------------------------------------------------------
ARTIFACTS_DIR: str = os.getenv("ARTIFACTS_DIR", "artifacts")

# ---------------------------------------------------------------------------
# Log alert thresholds
# ---------------------------------------------------------------------------
ERROR_COUNT_THRESHOLD: int = int(os.getenv("ERROR_COUNT_THRESHOLD", "50"))
WARNING_COUNT_THRESHOLD: int = int(os.getenv("WARNING_COUNT_THRESHOLD", "200"))
FATAL_COUNT_THRESHOLD: int = int(os.getenv("FATAL_COUNT_THRESHOLD", "5"))

# ---------------------------------------------------------------------------
# Retrieval
# ---------------------------------------------------------------------------
TOP_K_SIMILAR_TICKETS: int = int(os.getenv("TOP_K_SIMILAR_TICKETS", "5"))
TOP_K_LOG_LINES: int = int(os.getenv("TOP_K_LOG_LINES", "20"))
TOP_K_CONFLUENCE: int = int(os.getenv("TOP_K_CONFLUENCE", "3"))

# ChromaDB
CHROMA_PERSIST_DIR: str = os.getenv("CHROMA_PERSIST_DIR", "/tmp/aiops_chroma")

print("Configuration loaded.")

## 3. Celery Application (`celery_app.py`)

Celery application instance and task routing.

In [ ]:
"""Celery application instance and task routing."""

from celery import Celery

celery_app = Celery(
    "aiops",
    broker=REDIS_URL,
    backend=REDIS_URL,
    include=[
        "aiops.tasks.classify",
        "aiops.tasks.log_fetch",
        "aiops.tasks.search",
        "aiops.tasks.rca",
        "aiops.tasks.notify",
        "aiops.tasks.confluence_kb",
    ],
)

celery_app.conf.update(
    task_serializer="json",
    result_serializer="json",
    accept_content=["json"],
    timezone="UTC",
    enable_utc=True,
    task_acks_late=True,
    worker_prefetch_multiplier=1,
    task_routes={
        "aiops.tasks.classify.*": {"queue": "classify"},
        "aiops.tasks.log_fetch.*": {"queue": "logs"},
        "aiops.tasks.search.*": {"queue": "search"},
        "aiops.tasks.rca.*": {"queue": "rca"},
        "aiops.tasks.notify.*": {"queue": "notify"},
        "aiops.tasks.confluence_kb.*": {"queue": "confluence"},
    },
    task_default_retry_delay=30,   # seconds
    task_max_retries=3,
)

print("Celery app configured.")

## 4. Model Loader (`models/loader.py`)

Loads trained model artifacts once at process startup.

Artifacts expected under `ARTIFACTS_DIR`:
- `classification_deberta/` — DeBERTa seq-classification model + tokenizer + `label_encoder.joblib`
- `priority_lgbm/` — LightGBM booster (`priority_model.txt`) + `tfidf_vectorizer.joblib` + `priority_label_encoder.joblib` + `ohe_encoder.joblib`
- `log_reranker/` — cross-encoder checkpoint
- `incident_risk/` — LightGBM booster (`incident_risk_model.txt`)

In [ ]:
"""Load trained model artifacts once at process startup."""

import logging
from pathlib import Path
from typing import Any

import joblib

logger = logging.getLogger(__name__)

_REGISTRY: dict[str, Any] = {}


def _art(name: str) -> Path:
    return Path(ARTIFACTS_DIR) / name


def _load_category_model() -> None:
    """Load DeBERTa category classifier (optional — falls back to None)."""
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        import torch

        path = _art("classification_deberta")
        if not path.exists():
            logger.warning("Category model not found at %s — skipping", path)
            return

        tok = AutoTokenizer.from_pretrained(str(path))
        model = AutoModelForSequenceClassification.from_pretrained(str(path))
        model.eval()
        le = joblib.load(path / "label_encoder.joblib")

        _REGISTRY["cat_tokenizer"] = tok
        _REGISTRY["cat_model"] = model
        _REGISTRY["cat_le"] = le
        logger.info("Category model loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load category model: %s", exc)


def _load_priority_model() -> None:
    """Load LightGBM priority classifier."""
    try:
        import lightgbm as lgb

        path = _art("priority_lgbm")
        if not path.exists():
            logger.warning("Priority model not found at %s — skipping", path)
            return

        booster = lgb.Booster(model_file=str(path / "priority_model.txt"))
        tfidf = joblib.load(path / "tfidf_vectorizer.joblib")
        le = joblib.load(path / "priority_label_encoder.joblib")
        ohe = joblib.load(path / "ohe_encoder.joblib") if (path / "ohe_encoder.joblib").exists() else None

        _REGISTRY["prio_booster"] = booster
        _REGISTRY["prio_tfidf"] = tfidf
        _REGISTRY["prio_le"] = le
        _REGISTRY["prio_ohe"] = ohe
        logger.info("Priority model loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load priority model: %s", exc)


def _load_log_reranker() -> None:
    """Load cross-encoder log re-ranker."""
    try:
        from sentence_transformers.cross_encoder import CrossEncoder

        path = _art("log_reranker")
        if not path.exists():
            logger.warning("Log reranker not found at %s — skipping", path)
            return

        _REGISTRY["log_reranker"] = CrossEncoder(str(path))
        logger.info("Log reranker loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load log reranker: %s", exc)


def _load_incident_risk() -> None:
    """Load LightGBM incident-risk predictor."""
    try:
        import lightgbm as lgb

        path = _art("incident_risk")
        if not path.exists():
            logger.warning("Incident risk model not found at %s — skipping", path)
            return

        _REGISTRY["incident_booster"] = lgb.Booster(
            model_file=str(path / "incident_risk_model.txt")
        )
        logger.info("Incident risk model loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load incident risk model: %s", exc)


def load_all() -> None:
    """Call once at application startup to populate the model registry."""
    if _REGISTRY:
        return  # already loaded
    _load_category_model()
    _load_priority_model()
    _load_log_reranker()
    _load_incident_risk()
    logger.info("Model registry ready: %s", list(_REGISTRY.keys()))


def get_model(name: str) -> Any:
    """Retrieve a loaded artefact by name; returns None if unavailable."""
    return _REGISTRY.get(name)


# Load all models
load_all()
print("Model registry:", list(_REGISTRY.keys()) or "(no artifacts found)")

## 5. Classify Task (`tasks/classify.py`)

Classifies a Jira ticket (priority + category) and posts results back to Jira.

In [ ]:
"""Classify a Jira ticket (priority + category) and post back."""

import numpy as np


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _build_text(ticket: dict) -> str:
    parts = [
        ticket.get("summary", ""),
        ticket.get("description", ""),
        ticket.get("comments_text", ""),
        ticket.get("top_error_lines", ""),
    ]
    return " [SEP] ".join(p for p in parts if p)


def _classify_category(text: str) -> str | None:
    """Run DeBERTa category model; returns label string or None."""
    import torch

    model = get_model("cat_model")
    tok = get_model("cat_tokenizer")
    le = get_model("cat_le")
    if model is None or tok is None or le is None:
        return None

    try:
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=384)
        with torch.no_grad():
            logits = model(**inputs).logits
        idx = int(torch.argmax(logits, dim=-1).item())
        return str(le.inverse_transform([idx])[0])
    except Exception as exc:
        logger.warning("Category inference failed: %s", exc)
        return None


def _classify_priority(ticket: dict) -> str | None:
    """Run LightGBM priority model; returns P1–P5 string or None."""
    booster = get_model("prio_booster")
    tfidf = get_model("prio_tfidf")
    le = get_model("prio_le")
    if booster is None or tfidf is None or le is None:
        return None

    try:
        text = _build_text(ticket)
        tfidf_feat = tfidf.transform([text]).toarray()

        numeric_cols = [
            "affected_users", "downtime_minutes", "error_count",
            "fatal_count", "timeout_count", "auth_error_count",
        ]
        num_feat = np.array([[float(ticket.get(c, 0) or 0) for c in numeric_cols]])

        ohe = get_model("prio_ohe")
        cat_cols = ["env", "service"]
        cat_vals = [[str(ticket.get(c, "unknown")) for c in cat_cols]]
        if ohe is not None:
            cat_feat = ohe.transform(cat_vals)
        else:
            cat_feat = np.zeros((1, 1))

        X = np.hstack([tfidf_feat, num_feat, cat_feat])
        proba = booster.predict(X)
        idx = int(np.argmax(proba, axis=1)[0])
        return str(le.inverse_transform([idx])[0])
    except Exception as exc:
        logger.warning("Priority inference failed: %s", exc)
        return None


def _predict_incident_risk(ticket: dict) -> float:
    """Return probability [0,1] that this ticket precedes an incident."""
    booster = get_model("incident_booster")
    if booster is None:
        return 0.0
    try:
        tfidf = get_model("prio_tfidf")
        text = _build_text(ticket)
        tfidf_feat = tfidf.transform([text]).toarray() if tfidf else np.zeros((1, 100))
        numeric_cols = [
            "affected_users", "downtime_minutes", "error_count",
            "fatal_count", "timeout_count", "auth_error_count",
        ]
        num_feat = np.array([[float(ticket.get(c, 0) or 0) for c in numeric_cols]])
        X = np.hstack([tfidf_feat, num_feat])
        proba = booster.predict(X)
        return float(proba[0]) if proba.ndim == 1 else float(proba[0, 1])
    except Exception as exc:
        logger.warning("Incident risk inference failed: %s", exc)
        return 0.0


def _post_jira_comment(ticket_key: str, body: str) -> None:
    """Post a plain-text comment to a Jira issue."""
    import requests
    from requests.auth import HTTPBasicAuth

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}/comment"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    payload = {
        "body": {
            "type": "doc",
            "version": 1,
            "content": [
                {
                    "type": "paragraph",
                    "content": [{"type": "text", "text": body}],
                }
            ],
        }
    }
    resp = requests.post(url, json=payload, auth=auth, timeout=15)
    if not resp.ok:
        logger.error("Failed to post Jira comment: %s %s", resp.status_code, resp.text)


def _update_jira_fields(ticket_key: str, fields: dict) -> None:
    """Update arbitrary Jira issue fields."""
    import requests
    from requests.auth import HTTPBasicAuth

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    resp = requests.put(url, json={"fields": fields}, auth=auth, timeout=15)
    if not resp.ok:
        logger.error("Failed to update Jira fields: %s %s", resp.status_code, resp.text)


def classify_ticket(ticket: dict) -> dict:
    """Classify a ticket and post results back to Jira.

    Args:
        ticket: dict containing at minimum ``key``, ``summary``, ``description``.

    Returns:
        dict with ``category``, ``priority``, ``incident_risk``.
    """
    load_all()

    ticket_key = ticket.get("key", "UNKNOWN")
    logger.info("Classifying ticket %s", ticket_key)

    # Acknowledge receipt
    _post_jira_comment(
        ticket_key,
        "\U0001f916 AI Ops: Ticket received. Classification and RCA analysis in progress\u2026",
    )

    text = _build_text(ticket)
    category = _classify_category(text) or ticket.get("final_category", "Unknown")
    priority = _classify_priority(ticket) or ticket.get("final_priority", "P3")
    incident_risk = _predict_incident_risk(ticket)

    # Post classification summary
    comment = (
        f"\U0001f4ca *Classification Results*\n"
        f"\u2022 Category: {category}\n"
        f"\u2022 Priority: {priority}\n"
        f"\u2022 Incident Risk Score: {incident_risk:.0%}\n"
    )
    _post_jira_comment(ticket_key, comment)

    # Update Jira priority field if we have a valid mapping
    priority_map = {"P1": "Highest", "P2": "High", "P3": "Medium", "P4": "Low", "P5": "Lowest"}
    jira_priority = priority_map.get(priority, "Medium")
    _update_jira_fields(ticket_key, {"priority": {"name": jira_priority}})

    result = {"category": category, "priority": priority, "incident_risk": incident_risk}
    logger.info("Ticket %s classified: %s", ticket_key, result)
    return result


print("Classify functions defined.")

## 6. Log Fetch Task (`tasks/log_fetch.py`)

Fetches logs from Azure File Share (mobile) or Datadog (server) and re-ranks them.

In [ ]:
"""Fetch logs from Azure File Share (mobile) or Datadog (server)."""

import time
from datetime import datetime, timezone, timedelta


# ---------------------------------------------------------------------------
# Log re-ranking helper
# ---------------------------------------------------------------------------

def _rerank_logs(query: str, log_lines: list[str]) -> list[str]:
    """Score log lines against the ticket query and return top-k."""
    reranker = get_model("log_reranker")
    if reranker is None or not log_lines:
        return log_lines[:TOP_K_LOG_LINES]

    try:
        pairs = [(query, line) for line in log_lines]
        scores = reranker.predict(pairs)
        ranked = sorted(zip(scores, log_lines), key=lambda x: x[0], reverse=True)
        return [line for _, line in ranked[:TOP_K_LOG_LINES]]
    except Exception as exc:
        logger.warning("Log reranking failed: %s", exc)
        return log_lines[:TOP_K_LOG_LINES]


# ---------------------------------------------------------------------------
# Azure File Share — mobile logs
# ---------------------------------------------------------------------------

def _fetch_azure_logs(service: str, env: str, lookback_hours: int = 6) -> list[str]:
    """Fetch log lines from Azure File Share for mobile issues."""
    try:
        from azure.storage.fileshare import ShareServiceClient

        conn_str = AZURE_STORAGE_CONNECTION_STRING
        if not conn_str:
            logger.warning("AZURE_STORAGE_CONNECTION_STRING not set — skipping mobile logs")
            return []

        svc = ShareServiceClient.from_connection_string(conn_str)
        share_client = svc.get_share_client(AZURE_FILE_SHARE_NAME)

        directory = AZURE_LOG_DIRECTORY or service
        dir_client = share_client.get_directory_client(directory)

        cutoff = datetime.now(timezone.utc) - timedelta(hours=lookback_hours)
        lines: list[str] = []

        for item in dir_client.list_directories_and_files():
            if item["is_directory"]:
                continue
            if item.get("last_modified") and item["last_modified"] < cutoff:
                continue

            file_client = dir_client.get_file_client(item["name"])
            content = file_client.download_file().readall().decode("utf-8", errors="replace")
            lines.extend(content.splitlines())

        logger.info("Fetched %d log lines from Azure File Share (service=%s)", len(lines), service)
        return lines
    except Exception as exc:
        logger.error("Azure log fetch failed: %s", exc)
        return []


# ---------------------------------------------------------------------------
# Datadog — server logs
# ---------------------------------------------------------------------------

def _fetch_datadog_logs(service: str, env: str, lookback_hours: int | None = None) -> list[str]:
    """Fetch log lines from Datadog Logs API for server issues."""
    try:
        import requests

        if not DATADOG_API_KEY:
            logger.warning("DATADOG_API_KEY not set — skipping Datadog logs")
            return []

        hours = lookback_hours or DATADOG_LOG_LOOKBACK_HOURS
        now = datetime.now(timezone.utc)
        start = now - timedelta(hours=hours)

        url = f"https://api.{DATADOG_SITE}/api/v2/logs/events/search"
        headers = {
            "DD-API-KEY": DATADOG_API_KEY,
            "DD-APPLICATION-KEY": DATADOG_APP_KEY,
            "Content-Type": "application/json",
        }
        query_filter = f"service:{service}"
        if env:
            query_filter += f" env:{env}"
        query_filter += " status:(error OR warn OR critical)"

        payload = {
            "filter": {
                "query": query_filter,
                "from": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "to": now.strftime("%Y-%m-%dT%H:%M:%SZ"),
            },
            "sort": "timestamp",
            "page": {"limit": 1000},
        }

        resp = requests.post(url, json=payload, headers=headers, timeout=20)
        resp.raise_for_status()

        data = resp.json()
        events = data.get("data", [])
        lines: list[str] = []
        for evt in events:
            msg = evt.get("attributes", {}).get("message", "")
            if msg:
                lines.append(msg)

        logger.info("Fetched %d log lines from Datadog (service=%s)", len(lines), service)
        return lines
    except Exception as exc:
        logger.error("Datadog log fetch failed: %s", exc)
        return []


# ---------------------------------------------------------------------------
# Main fetch function
# ---------------------------------------------------------------------------

def fetch_logs(ticket: dict) -> dict:
    """Fetch and re-rank logs relevant to the ticket.

    Determines source (Azure = mobile, Datadog = server) from ticket category.

    Args:
        ticket: dict with ``key``, ``category``, ``service``, ``env``.

    Returns:
        dict with ``top_log_lines`` (list[str]) and ``source`` ("azure" | "datadog" | "none").
    """
    load_all()

    category = (ticket.get("category") or ticket.get("final_category") or "").lower()
    service = ticket.get("service", "")
    env = ticket.get("env", "")
    query = f"{ticket.get('summary', '')} {ticket.get('description', '')}"

    is_mobile = any(kw in category for kw in ("mobile", "android", "ios", "app"))

    if is_mobile:
        raw_lines = _fetch_azure_logs(service, env)
        source = "azure"
    else:
        raw_lines = _fetch_datadog_logs(service, env)
        source = "datadog"

    if not raw_lines:
        return {"top_log_lines": [], "source": "none"}

    top_lines = _rerank_logs(query, raw_lines)

    # Threshold monitoring: count errors/warnings
    error_count = sum(1 for l in raw_lines if "error" in l.lower() or "exception" in l.lower())
    warn_count = sum(1 for l in raw_lines if "warn" in l.lower())
    fatal_count = sum(1 for l in raw_lines if "fatal" in l.lower() or "critical" in l.lower())

    thresholds_exceeded = (
        error_count >= ERROR_COUNT_THRESHOLD
        or warn_count >= WARNING_COUNT_THRESHOLD
        or fatal_count >= FATAL_COUNT_THRESHOLD
    )

    if thresholds_exceeded:
        logger.warning(
            "Log thresholds exceeded for %s — errors=%d, warnings=%d, fatals=%d",
            ticket.get("key"), error_count, warn_count, fatal_count,
        )

    return {
        "top_log_lines": top_lines,
        "source": source,
        "error_count": error_count,
        "warn_count": warn_count,
        "fatal_count": fatal_count,
    }


print("Log fetch functions defined.")

## 7. Search Task (`tasks/search.py`)

Finds similar historical tickets (via ChromaDB) and relevant Confluence pages.

In [ ]:
"""Find similar historical tickets and relevant Confluence pages."""

import re


# ---------------------------------------------------------------------------
# Similar ticket retrieval via ChromaDB
# ---------------------------------------------------------------------------

def _get_chroma_collection():
    """Return (or lazily create) a persistent ChromaDB collection of historical tickets."""
    try:
        import chromadb

        persist_path = CHROMA_PERSIST_DIR
        client = chromadb.PersistentClient(path=persist_path)
        return client.get_or_create_collection("tickets")
    except Exception as exc:
        logger.warning("ChromaDB unavailable: %s", exc)
        return None


def _search_similar_tickets(query: str, top_k: int) -> list[dict]:
    """Query ChromaDB for the top-k most similar historical tickets."""
    collection = _get_chroma_collection()
    if collection is None:
        return []

    try:
        results = collection.query(
            query_texts=[query],
            n_results=top_k,
            include=["documents", "metadatas", "distances"],
        )
        tickets = []
        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]
        dists = results.get("distances", [[]])[0]
        for doc, meta, dist in zip(docs, metas, dists):
            tickets.append(
                {
                    "text": doc,
                    "metadata": meta,
                    "similarity": round(1 - dist, 4),
                }
            )
        return tickets
    except Exception as exc:
        logger.error("Similar ticket search failed: %s", exc)
        return []


def index_ticket(ticket: dict) -> None:
    """Upsert a ticket into the ChromaDB collection (call after resolution)."""
    collection = _get_chroma_collection()
    if collection is None:
        return
    try:
        text = (
            f"{ticket.get('summary', '')} "
            f"{ticket.get('description', '')} "
            f"{ticket.get('comments_text', '')}"
        )
        collection.upsert(
            ids=[ticket.get("key", "")],
            documents=[text],
            metadatas=[
                {
                    "key": ticket.get("key", ""),
                    "category": ticket.get("category", ""),
                    "priority": ticket.get("priority", ""),
                    "service": ticket.get("service", ""),
                }
            ],
        )
    except Exception as exc:
        logger.error("Failed to index ticket: %s", exc)


# ---------------------------------------------------------------------------
# Confluence search
# ---------------------------------------------------------------------------

def _search_confluence(query: str, top_k: int) -> list[dict]:
    """Search Confluence using CQL and return top-k matching pages."""
    try:
        import requests
        from requests.auth import HTTPBasicAuth

        if not CONFLUENCE_BASE_URL or not JIRA_API_TOKEN:
            logger.warning("Confluence credentials not configured — skipping")
            return []

        url = f"{CONFLUENCE_BASE_URL}/wiki/rest/api/content/search"
        auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
        cql = (
            f'space = "{CONFLUENCE_SPACE_KEY}" AND text ~ "{query}" '
            f'ORDER BY relevance DESC'
        )
        params = {
            "cql": cql,
            "limit": top_k,
            "expand": "body.storage,metadata.labels",
        }
        resp = requests.get(url, params=params, auth=auth, timeout=15)
        resp.raise_for_status()

        results = resp.json().get("results", [])
        pages = []
        for r in results:
            body_val = (
                r.get("body", {}).get("storage", {}).get("value", "")
            )
            # Strip HTML tags for plain-text summary
            plain = re.sub(r"<[^>]+>", " ", body_val)[:1000]
            pages.append(
                {
                    "title": r.get("title", ""),
                    "url": (
                        CONFLUENCE_BASE_URL
                        + r.get("_links", {}).get("webui", "")
                    ),
                    "excerpt": plain.strip(),
                }
            )
        logger.info("Confluence returned %d pages for query '%s'", len(pages), query[:60])
        return pages
    except Exception as exc:
        logger.error("Confluence search failed: %s", exc)
        return []


def search_context(ticket: dict) -> dict:
    """Find similar tickets and Confluence articles for the given ticket.

    Args:
        ticket: dict with ``key``, ``summary``, ``description``.

    Returns:
        dict with ``similar_tickets`` (list) and ``confluence_pages`` (list).
    """
    query = f"{ticket.get('summary', '')} {ticket.get('description', '')}".strip()
    if not query:
        return {"similar_tickets": [], "confluence_pages": []}

    similar = _search_similar_tickets(query, TOP_K_SIMILAR_TICKETS)
    confluence = _search_confluence(query[:200], TOP_K_CONFLUENCE)

    logger.info(
        "Ticket %s: found %d similar tickets, %d Confluence pages",
        ticket.get("key"), len(similar), len(confluence),
    )
    return {"similar_tickets": similar, "confluence_pages": confluence}


print("Search functions defined.")

## 8. RCA Task (`tasks/rca.py`)

Generates a Root Cause Analysis via LLM and posts it to Jira.

In [ ]:
"""Generate Root Cause Analysis via LLM and post to Jira."""


# ---------------------------------------------------------------------------
# LLM client helper
# ---------------------------------------------------------------------------

def _call_llm(prompt: str) -> str:
    """Call OpenAI or Azure OpenAI and return the response text."""
    # Prefer Azure OpenAI if endpoint is configured, otherwise fall back to OpenAI
    if AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY:
        return _call_azure_openai(prompt)
    if OPENAI_API_KEY:
        return _call_openai(prompt)
    raise RuntimeError("No LLM credentials configured (OPENAI_API_KEY or AZURE_OPENAI_*).")


def _call_openai(prompt: str) -> str:
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=2048,
    )
    return resp.choices[0].message.content.strip()


def _call_azure_openai(prompt: str) -> str:
    from openai import AzureOpenAI

    client = AzureOpenAI(
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
    )
    resp = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=2048,
    )
    return resp.choices[0].message.content.strip()


# ---------------------------------------------------------------------------
# RCA prompt builder
# ---------------------------------------------------------------------------

def _build_rca_prompt(
    ticket: dict,
    similar_tickets: list[dict],
    confluence_pages: list[dict],
    top_log_lines: list[str],
    incident_risk: float,
) -> str:
    similar_text = "\n".join(
        f"- [{t.get('metadata', {}).get('key', 'N/A')}] "
        f"(similarity {t.get('similarity', 0):.0%}): {t.get('text', '')[:300]}"
        for t in similar_tickets
    ) or "None found."

    confluence_text = "\n".join(
        f"- [{p.get('title', '')}]({p.get('url', '')}): {p.get('excerpt', '')[:300]}"
        for p in confluence_pages
    ) or "None found."

    logs_text = "\n".join(top_log_lines[:20]) or "No logs retrieved."

    return f"""You are an expert Site Reliability Engineer performing a Root Cause Analysis.

## Ticket Details
**Key**: {ticket.get('key', 'N/A')}
**Summary**: {ticket.get('summary', '')}
**Description**: {ticket.get('description', '')}
**Category**: {ticket.get('category', '')}
**Priority**: {ticket.get('priority', '')}
**Service**: {ticket.get('service', '')}
**Environment**: {ticket.get('env', '')}
**Incident Risk Score**: {incident_risk:.0%}

## Similar Past Tickets
{similar_text}

## Relevant Confluence Knowledge Base
{confluence_text}

## Top Relevant Log Lines
{logs_text}

## Instructions
Produce a structured RCA with the following sections:

1. **Root Cause** — What is the primary technical cause?
2. **Contributing Factors** — Secondary conditions that amplified the issue.
3. **Impact** — Services/users affected and severity.
4. **Timeline** — Inferred sequence of events leading to the issue.
5. **Resolution Steps** — Concrete actions to resolve the issue now.
6. **Prevention / Follow-up** — Long-term fixes, monitoring improvements, runbook updates.

Be concise, technical, and actionable. Base your analysis strictly on the evidence provided.
"""


# ---------------------------------------------------------------------------
# Jira comment helper
# ---------------------------------------------------------------------------

def _post_rca_jira_comment(ticket_key: str, body: str) -> None:
    import requests
    from requests.auth import HTTPBasicAuth

    if not JIRA_BASE_URL or not JIRA_API_TOKEN:
        logger.warning("Jira credentials not configured — skipping comment post")
        return

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}/comment"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    payload = {
        "body": {
            "type": "doc",
            "version": 1,
            "content": [
                {
                    "type": "paragraph",
                    "content": [{"type": "text", "text": body}],
                }
            ],
        }
    }
    resp = requests.post(url, json=payload, auth=auth, timeout=15)
    if not resp.ok:
        logger.error("Failed to post Jira comment: %s %s", resp.status_code, resp.text)


# ---------------------------------------------------------------------------
# RCA generation
# ---------------------------------------------------------------------------

def generate_rca(ticket: dict, context: dict) -> dict:
    """Generate an RCA and post it to Jira.

    Args:
        ticket: classified ticket dict (key, summary, description, category, priority, …).
        context: combined output of classify, log_fetch, and search tasks:
            {similar_tickets, confluence_pages, top_log_lines, incident_risk}.

    Returns:
        dict with ``rca_text`` and ``confluence_created`` flag.
    """
    ticket_key = ticket.get("key", "UNKNOWN")
    logger.info("Generating RCA for %s", ticket_key)

    similar = context.get("similar_tickets", [])
    confluence = context.get("confluence_pages", [])
    logs = context.get("top_log_lines", [])
    incident_risk = float(context.get("incident_risk", 0.0))

    prompt = _build_rca_prompt(ticket, similar, confluence, logs, incident_risk)
    rca_text = _call_llm(prompt)

    # Post RCA to Jira
    header = (
        f"\U0001f50d *Root Cause Analysis \u2014 {ticket_key}*\n"
        f"\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n"
    )
    _post_rca_jira_comment(ticket_key, header + rca_text)

    # If no Confluence pages found, create a knowledge article
    confluence_created = False
    if not confluence:
        logger.info("No Confluence pages found — creating knowledge article for %s", ticket_key)
        create_kb_article(ticket=ticket, rca_text=rca_text)
        confluence_created = True

    return {"rca_text": rca_text, "confluence_created": confluence_created}


print("RCA functions defined.")

## 9. Notify Task (`tasks/notify.py`)

Sends email alerts for log thresholds and Datadog monitor breaches.

In [ ]:
"""Send email alerts for log thresholds and Datadog monitor breaches."""

import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText


# ---------------------------------------------------------------------------
# SMTP helper
# ---------------------------------------------------------------------------

def _send_email(subject: str, body_html: str, recipients: list[str] | None = None) -> None:
    to_list = recipients or ALERT_EMAIL_RECIPIENTS
    if not to_list:
        logger.warning("No email recipients configured — skipping email")
        return
    if not SMTP_USER or not SMTP_PASSWORD:
        logger.warning("SMTP credentials not configured — skipping email")
        return

    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"] = SMTP_USER
    msg["To"] = ", ".join(to_list)
    msg.attach(MIMEText(body_html, "html"))

    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
            server.ehlo()
            server.starttls()
            server.login(SMTP_USER, SMTP_PASSWORD)
            server.sendmail(SMTP_USER, to_list, msg.as_string())
        logger.info("Email sent: %s -> %s", subject, to_list)
    except Exception as exc:
        logger.error("Failed to send email: %s", exc)
        raise


# ---------------------------------------------------------------------------
# Notification functions
# ---------------------------------------------------------------------------

def send_threshold_alert(
    ticket: dict,
    error_count: int,
    warn_count: int,
    fatal_count: int,
    top_lines: list[str],
) -> None:
    """Send an email alert when log thresholds are exceeded.

    Args:
        ticket: ticket dict (key, summary, service, env).
        error_count: number of error-level log lines.
        warn_count: number of warning-level log lines.
        fatal_count: number of fatal/critical log lines.
        top_lines: most relevant log excerpts.
    """
    ticket_key = ticket.get("key", "N/A")
    service = ticket.get("service", "unknown")
    env = ticket.get("env", "unknown")

    subject = f"\u26a0\ufe0f AI Ops Alert: Log Threshold Exceeded \u2014 {ticket_key} [{service}/{env}]"

    lines_html = "".join(
        f"<li><code>{line[:200]}</code></li>" for line in top_lines
    )

    body = f"""
    <html><body>
    <h2>\U0001f6a8 Log Threshold Alert</h2>
    <p><strong>Ticket:</strong> {ticket_key} \u2014 {ticket.get('summary', '')}</p>
    <p><strong>Service:</strong> {service} | <strong>Env:</strong> {env}</p>
    <table border="1" cellpadding="6" cellspacing="0">
      <tr><th>Metric</th><th>Count</th><th>Threshold</th></tr>
      <tr><td>Errors</td><td>{error_count}</td><td>{ERROR_COUNT_THRESHOLD}</td></tr>
      <tr><td>Warnings</td><td>{warn_count}</td><td>{WARNING_COUNT_THRESHOLD}</td></tr>
      <tr><td>Fatals</td><td>{fatal_count}</td><td>{FATAL_COUNT_THRESHOLD}</td></tr>
    </table>
    <h3>Top Log Lines</h3>
    <ul>{lines_html}</ul>
    <p>Please investigate immediately.</p>
    </body></html>
    """

    _send_email(subject, body)


def send_rca_notification(ticket: dict, rca_text: str) -> None:
    """Email the generated RCA to stakeholders.

    Args:
        ticket: ticket dict.
        rca_text: full RCA markdown text.
    """
    import html as _html

    ticket_key = ticket.get("key", "N/A")
    subject = f"\U0001f4cb AI Ops RCA Ready \u2014 {ticket_key}: {ticket.get('summary', '')[:80]}"

    rca_html = _html.escape(rca_text).replace("\n", "<br>")
    body = f"""
    <html><body>
    <h2>Root Cause Analysis \u2014 {ticket_key}</h2>
    <p><strong>Summary:</strong> {ticket.get('summary', '')}</p>
    <p><strong>Priority:</strong> {ticket.get('priority', '')} |
       <strong>Category:</strong> {ticket.get('category', '')}</p>
    <hr/>
    {rca_html}
    <hr/>
    <p><em>Generated by AI Ops Automation System</em></p>
    </body></html>
    """

    _send_email(subject, body)


def send_datadog_alert(alert_payload: dict) -> None:
    """Email stakeholders when Datadog signals a monitor threshold breach.

    Args:
        alert_payload: parsed Datadog webhook payload.
    """
    monitor_name = alert_payload.get("monitor_name", "Unknown Monitor")
    metric = alert_payload.get("metric", "")
    value = alert_payload.get("value", "")
    threshold = alert_payload.get("threshold", "")
    screenshot_url = alert_payload.get("snapshot_url", "")
    transition = alert_payload.get("transition", "ALERT")

    subject = f"\U0001f534 Datadog Monitor {transition}: {monitor_name}"

    screenshot_html = (
        f'<p><a href="{screenshot_url}">View Snapshot</a></p>'
        if screenshot_url
        else ""
    )

    body = f"""
    <html><body>
    <h2>\U0001f534 Datadog Monitor Alert</h2>
    <p><strong>Monitor:</strong> {monitor_name}</p>
    <p><strong>Metric:</strong> {metric}</p>
    <p><strong>Current Value:</strong> {value} | <strong>Threshold:</strong> {threshold}</p>
    <p><strong>Transition:</strong> {transition}</p>
    {screenshot_html}
    <pre>{str(alert_payload)[:2000]}</pre>
    <p><em>AI Ops has queued an RCA for this event.</em></p>
    </body></html>
    """

    _send_email(subject, body)


print("Notification functions defined.")

## 10. Confluence KB Task (`tasks/confluence_kb.py`)

Creates Confluence knowledge articles from AI-generated RCAs.

In [ ]:
"""Create Confluence knowledge articles from AI-generated RCAs."""


def _markdown_to_confluence_storage(md: str) -> str:
    """Very lightweight Markdown → Confluence Storage Format conversion."""
    lines = md.split("\n")
    out: list[str] = []
    for line in lines:
        # Headings
        if line.startswith("### "):
            out.append(f"<h3>{line[4:]}</h3>")
        elif line.startswith("## "):
            out.append(f"<h2>{line[3:]}</h2>")
        elif line.startswith("# "):
            out.append(f"<h1>{line[2:]}</h1>")
        # Bold
        else:
            line = re.sub(r"\*\*(.*?)\*\*", r"<strong>\1</strong>", line)
            # Bullet list
            if line.startswith("- "):
                out.append(f"<li>{line[2:]}</li>")
            else:
                out.append(f"<p>{line}</p>" if line.strip() else "<p> </p>")

    return "\n".join(out)


def create_kb_article(ticket: dict, rca_text: str) -> dict:
    """Create a new Confluence page from the RCA when no KB article exists.

    Args:
        ticket: ticket dict (key, summary, category, service).
        rca_text: full RCA text generated by the LLM.

    Returns:
        dict with ``page_id`` and ``page_url``.
    """
    import requests
    from requests.auth import HTTPBasicAuth

    if not CONFLUENCE_BASE_URL or not JIRA_API_TOKEN:
        logger.warning("Confluence credentials not configured — skipping KB creation")
        return {"page_id": None, "page_url": None}

    ticket_key = ticket.get("key", "N/A")
    summary = ticket.get("summary", "Untitled")
    category = ticket.get("category", "General")
    service = ticket.get("service", "unknown")

    title = f"[RCA] {ticket_key} \u2014 {summary[:80]}"
    storage_body = _markdown_to_confluence_storage(rca_text)

    # Wrap in a Confluence info macro header
    page_body = f"""
<ac:structured-macro ac:name="info">
  <ac:rich-text-body>
    <p>Auto-generated RCA by AI Ops for Jira ticket
       <strong>{ticket_key}</strong> ({category} / {service}).
    </p>
  </ac:rich-text-body>
</ac:structured-macro>
{storage_body}
"""

    url = f"{CONFLUENCE_BASE_URL}/wiki/rest/api/content"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)

    payload: dict = {
        "type": "page",
        "title": title,
        "space": {"key": CONFLUENCE_SPACE_KEY},
        "body": {
            "storage": {
                "value": page_body,
                "representation": "storage",
            }
        },
    }
    if CONFLUENCE_PARENT_PAGE_ID:
        payload["ancestors"] = [{"id": CONFLUENCE_PARENT_PAGE_ID}]

    resp = requests.post(url, json=payload, auth=auth, timeout=20)
    resp.raise_for_status()
    data = resp.json()
    page_id = data.get("id")
    webui = data.get("_links", {}).get("webui", "")
    page_url = CONFLUENCE_BASE_URL + webui

    logger.info("Created Confluence page '%s' (id=%s)", title, page_id)

    # Link the new Confluence page back to the Jira ticket
    _link_confluence_to_jira(ticket_key, page_url, title)

    return {"page_id": page_id, "page_url": page_url}


def _link_confluence_to_jira(ticket_key: str, page_url: str, page_title: str) -> None:
    """Add a remote link on the Jira issue pointing to the new Confluence page."""
    import requests
    from requests.auth import HTTPBasicAuth

    if not JIRA_BASE_URL or not JIRA_API_TOKEN:
        return

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}/remotelink"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    payload = {
        "globalId": f"confluence-rca-{ticket_key}",
        "object": {
            "url": page_url,
            "title": page_title,
            "icon": {
                "url16x16": "https://confluence.atlassian.com/images/logo/confluence_16.png",
                "title": "Confluence Page",
            },
        },
    }
    resp = requests.post(url, json=payload, auth=auth, timeout=15)
    if not resp.ok:
        logger.warning(
            "Could not link Confluence page to Jira %s: %s", ticket_key, resp.text
        )


print("Confluence KB functions defined.")

## 11. FastAPI Application (`main.py`)

FastAPI application with Jira & Datadog webhook endpoints and pipeline orchestration.

In [ ]:
"""FastAPI application: Jira & Datadog webhook endpoints + pipeline orchestration."""

from typing import Any
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse


app = FastAPI(
    title="AI Ops Automation",
    description=(
        "End-to-end AI Ops pipeline: Jira ingestion → classification → "
        "log fetching → RCA generation → notifications."
    ),
    version="1.0.0",
)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _extract_description(desc: Any) -> str:
    """Extract plain text from Jira's Atlassian Document Format description."""
    if desc is None:
        return ""
    if isinstance(desc, str):
        return desc
    # ADF: traverse content nodes
    parts: list[str] = []

    def _walk(node: Any) -> None:
        if isinstance(node, dict):
            if node.get("type") == "text":
                parts.append(node.get("text", ""))
            for child in node.get("content", []):
                _walk(child)
        elif isinstance(node, list):
            for item in node:
                _walk(item)

    _walk(desc)
    return " ".join(parts).strip()


def _extract_custom_field(fields: dict, field_id: str, default: Any) -> Any:
    val = fields.get(field_id)
    if val is None:
        return default
    if isinstance(val, dict):
        return val.get("value", default)
    return val


def _create_jira_ticket_for_datadog(payload: dict) -> str | None:
    """Create a Jira issue for a Datadog monitor alert; return the issue key."""
    try:
        import requests
        from requests.auth import HTTPBasicAuth

        if not JIRA_BASE_URL or not JIRA_API_TOKEN:
            return None

        monitor_name = payload.get("monitor_name", "Datadog Monitor Alert")
        url = f"{JIRA_BASE_URL}/rest/api/3/issue"
        auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)

        issue_payload = {
            "fields": {
                "project": {"key": JIRA_PROJECT_KEY},
                "summary": f"[Datadog Alert] {monitor_name}",
                "issuetype": {"name": "Incident"},
                "priority": {"name": "High"},
                "description": {
                    "type": "doc",
                    "version": 1,
                    "content": [
                        {
                            "type": "paragraph",
                            "content": [
                                {
                                    "type": "text",
                                    "text": (
                                        f"Automated Jira ticket created by AI Ops.\n"
                                        f"Monitor: {monitor_name}\n"
                                        f"Metric: {payload.get('metric', 'N/A')}\n"
                                        f"Value: {payload.get('value', 'N/A')}\n"
                                        f"Threshold: {payload.get('threshold', 'N/A')}"
                                    ),
                                }
                            ],
                        }
                    ],
                },
            }
        }

        resp = requests.post(url, json=issue_payload, auth=auth, timeout=15)
        if resp.ok:
            key = resp.json().get("key")
            logger.info("Created Jira ticket %s for Datadog alert", key)
            return key
        else:
            logger.error("Failed to create Jira ticket: %s %s", resp.status_code, resp.text)
            return None
    except Exception as exc:
        logger.error("Exception creating Jira ticket: %s", exc)
        return None


def _run_pipeline(ticket: dict) -> None:
    """Run the full analysis pipeline synchronously (no Celery in notebook context)."""
    # Step 1: Classify
    classify_result = classify_ticket(ticket)
    merged = {**ticket, **classify_result}

    # Step 2: Parallel context (log fetch + search)
    log_result = fetch_logs(merged)
    search_result = search_context(merged)
    context = {**log_result, **search_result}
    context["incident_risk"] = merged.get("incident_risk", 0.0)

    # Step 3: RCA
    rca_result = generate_rca(merged, context)

    # Step 4: Notify
    send_rca_notification(ticket=merged, rca_text=rca_result.get("rca_text", ""))

    return rca_result


print("FastAPI app and pipeline helpers defined.")

## 12. Manual Pipeline Trigger (Testing)

Use this cell to manually run the full pipeline with a sample ticket — useful for testing without a live Jira webhook.

In [ ]:
# Example: manually trigger the pipeline with a sample ticket
sample_ticket = {
    "key": "OPS-123",
    "summary": "Payment service timing out for EU users",
    "description": "Users in the EU region are experiencing 504 gateway timeouts on /api/payments. Error rate spiked to 15% at 14:30 UTC.",
    "comments_text": "",
    "top_error_lines": "",
    "env": "production",
    "service": "payment-service",
    "affected_users": 5000,
    "downtime_minutes": 30,
    "error_count": 120,
    "fatal_count": 2,
    "timeout_count": 95,
    "auth_error_count": 0,
}

# Uncomment to run the full pipeline:
# result = _run_pipeline(sample_ticket)
# print(result)

# Or run individual steps:
# classify_result = classify_ticket(sample_ticket)
# print("Classification:", classify_result)

# search_result = search_context(sample_ticket)
# print("Search:", search_result)

# log_result = fetch_logs(sample_ticket)
# print("Logs:", log_result)

print("Sample ticket ready. Uncomment the lines above to run the pipeline.")